# 🎓 MCQ Generator — Qwen3-4B-Base (No RAG)

## How It Works
1. Lecture JSON files are loaded from `/kaggle/input/datasets/mayarboghdady/lec-nots/`
2. All lecture content is loaded, slide IDs resolved, and text truncated to fit model context
3. A structured prompt with few-shot examples is built
4. `Qwen/Qwen3-4B-Base` generates 100 MCQs directly from the lecture content
5. Output is validated and saved as `mcqs_output.json`

## Changes in This Version
- ✅ Fixed path to `/kaggle/input/datasets/mayarboghdady/lec-nots`
- ✅ Fixed slide ID parsing — handles `slide_title` and `slide_paragraph` keys
- ✅ Added context length truncation (120K chars) to fit Qwen3 32K token limit
- ✅ Auto-detects JSON structure regardless of key names

In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [2]:
import os
import json
import re
import glob
import torch
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}         : {torch.cuda.get_device_name(i)}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
  GPU 0         : Tesla T4
  GPU 1         : Tesla T4


In [3]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME         = "Qwen/Qwen3-4B-Base"
MAX_NEW_TOKENS     = 4000
TEMPERATURE        = 0.7
TOP_P              = 0.9
REPETITION_PENALTY = 1.15

LECTURE_INPUT_DIR  = "/kaggle/input/datasets/mayarboghdady/lec-nots"
OUTPUT_PATH        = "/kaggle/working/mcqs_output.json"

# Keep lecture SHORT — model needs token budget to generate JSON
MAX_LECTURE_CHARS  = 8_000   # ~1800 tokens for lecture content

RUN_NUMBER = 1

LECTURE_SLICE_MAP = {
    1 : (0,       8_000),
    2 : (8_000,   16_000),
    3 : (16_000,  24_000),
    4 : (24_000,  32_000),
    5 : (32_000,  40_000),
    6 : (40_000,  48_000),
    7 : (48_000,  56_000),
    8 : (56_000,  64_000),
    9 : (64_000,  72_000),
    10: (72_000,  80_000),
}
SLICE_START, SLICE_END = LECTURE_SLICE_MAP[RUN_NUMBER]
OUTPUT_PATH = f"/kaggle/working/mcqs_run{RUN_NUMBER}.json"

NUM_QUESTIONS = 10

DIFFICULTY_DIST = {"Beginner": 3, "Intermediate": 4, "Difficult": 3}
BLOOM_DIST      = {"Remember": 2, "Understand": 2, "Apply": 2,
                   "Analyze": 2, "Evaluate": 1, "Create": 1}

print(f"Run {RUN_NUMBER}/10 | Chars {SLICE_START:,}–{SLICE_END:,} | {NUM_QUESTIONS} questions")

Run 1/10 | Chars 0–8,000 | 10 questions


In [4]:
import os, glob, json
from pathlib import Path


def find_json_files(start_dir: str) -> list:
    """
    Finds all JSON files starting from start_dir.
    Falls back to scanning all of /kaggle/input/ if none found.
    """
    files = sorted(glob.glob(str(Path(start_dir) / "**/*.json"), recursive=True))
    if files:
        return files

    print(f"No JSON files at {start_dir}. Auto-scanning /kaggle/input/ ...")
    print("\nDirectories found:")
    for entry in sorted(os.listdir("/kaggle/input")):
        print(f"  /kaggle/input/{entry}")

    all_json = sorted(glob.glob("/kaggle/input/**/*.json", recursive=True))
    if all_json:
        print(f"\nFound {len(all_json)} JSON file(s):")
        for f in all_json:
            print(f"  {f}")
        return all_json

    raise FileNotFoundError(
        "No JSON files found under /kaggle/input/.\n"
        "Add your dataset: Notebook → Data → Add Dataset"
    )


def extract_slide_content(item: dict, file_name: str, index: int) -> tuple:
    """
    Extracts slide_id, title, and content from a slide dict.
    Handles your actual JSON structure: slide_title + slide_paragraph.
    """
    # ── Slide ID ──
    slide_id = (
        item.get("slide_id")  or
        item.get("id")        or
        item.get("slide")     or
        item.get("slide_num") or
        item.get("number")    or
        f"{file_name}-S{index + 1}"
    )

    # ── Title ──
    title = (
        item.get("title")       or
        item.get("heading")     or
        item.get("slide_title") or   # your actual key
        item.get("name")        or
        ""
    )

    # ── Content ──
    content = (
        item.get("content")         or
        item.get("text")            or
        item.get("body")            or
        item.get("slide_paragraph") or   # your actual key
        item.get("description")     or
        item.get("explanation")     or
        str(item)
    )

    return str(slide_id), str(title), str(content)


def load_lecture_notes(input_dir: str) -> tuple:
    json_files     = find_json_files(input_dir)
    all_text_parts = []
    all_slide_ids  = []

    print(f"\nLoading {len(json_files)} JSON file(s):")
    for fpath in json_files:
        print(f"  {fpath}")

    for fpath in json_files:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        file_name = Path(fpath).stem

        # ── Format 1: List of slide objects ──
        if isinstance(data, list):
            all_text_parts.append(f"\n=== Lecture File: {file_name} ===\n")
            for i, item in enumerate(data):
                if isinstance(item, dict):
                    slide_id, title, content = extract_slide_content(item, file_name, i)
                    all_slide_ids.append(slide_id)
                    part = f"[Slide {slide_id}]"
                    if title and title != "None":
                        part += f" {title}\n"
                    part += f"\n{content}\n"
                    all_text_parts.append(part)
                elif isinstance(item, str):
                    all_text_parts.append(item + "\n")

        # ── Format 2: Dict with 'slides' key ──
        elif isinstance(data, dict) and "slides" in data:
            lecture_title = data.get("title") or data.get("lecture") or file_name
            all_text_parts.append(f"\n=== Lecture: {lecture_title} ===\n")
            for i, item in enumerate(data["slides"]):
                slide_id, title, content = extract_slide_content(item, file_name, i)
                all_slide_ids.append(slide_id)
                part = f"[Slide {slide_id}]"
                if title and title != "None":
                    part += f" {title}\n"
                part += f"\n{content}\n"
                all_text_parts.append(part)

        # ── Format 3: Single slide dict ──
        elif isinstance(data, dict):
            slide_id, title, content = extract_slide_content(data, file_name, 0)
            all_slide_ids.append(slide_id)
            part = f"[Slide {slide_id}]"
            if title and title != "None":
                part += f" {title}\n"
            part += f"\n{content}\n"
            all_text_parts.append(part)

        # ── Format 4: Raw string ──
        elif isinstance(data, str):
            all_text_parts.append(data + "\n")

    full_text = "\n".join(all_text_parts)

    print(f"\nTotal slides loaded : {len(all_slide_ids)}")
    print(f"Total characters    : {len(full_text):,}")
    print(f"Estimated words     : {len(full_text.split()):,}")

    # ── Slice for this run ──
    full_text = full_text[SLICE_START:SLICE_END]
    print(f"\nRun {RUN_NUMBER}: using chars {SLICE_START:,} → {SLICE_END:,}")
    print(f"Characters in slice : {len(full_text):,}")
    print(f"Estimated tokens    : ~{len(full_text.split()):,} words")

    # ── Safety check ──
    if len(full_text) == 0:
        raise ValueError(
            f"Slice {SLICE_START}:{SLICE_END} returned empty text.\n"
            f"Total lecture text may be shorter than expected.\n"
            f"Reduce RUN_NUMBER or check total chars above."
        )

    return full_text, all_slide_ids


LECTURE_TEXT, SLIDE_IDS = load_lecture_notes(LECTURE_INPUT_DIR)

print("\nFirst 500 characters of slice:")
print("-" * 60)
print(LECTURE_TEXT[:500])
print("-" * 60)
print(f"\nSample slide IDs : {SLIDE_IDS[:3]}")
print(f"Ready for Run {RUN_NUMBER}/10")


Loading 9 JSON file(s):
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_1_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_2_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_3_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_4_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_5_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_6_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_7_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_8_slide_explanations.json
  /kaggle/input/datasets/mayarboghdady/lec-nots/llm_lecture_9_slide_explanations.json

Total slides loaded : 664
Total characters    : 597,778
Estimated words     : 87,220

Run 1: using chars 0 → 8,000
Characters in slice : 8,000
Estimated tokens    : ~1,199 words

First 500 characters 

In [5]:
# ─────────────────────────────────────────────────────────────────
# Cell 5 — Few-Shot Examples
# Teaches the model the EXACT output format and quality required
# ─────────────────────────────────────────────────────────────────

FEW_SHOT_EXAMPLES = """
[
  {
    "question": "Which component is responsible for converting text into numerical representations that a neural network can process?",
    "question_type": "single_answer",
    "options": {
      "A": "Attention mechanism",
      "B": "Tokenizer",
      "C": "Optimizer",
      "D": "Decoder"
    },
    "correct_answer": "B. Tokenizer",
    "correct_letter": "B",
    "difficulty": "Beginner",
    "bloom_level": "Remember",
    "tested_concepts": "Tokenization; text preprocessing",
    "evidence_slide_ids": "llm_lecture_1_slide_explanations-S3",
    "why_correct": "The tokenizer converts raw text into tokens that can be processed by the model.",
    "why_each_wrong_option_is_plausible": "A is related to processing tokens after tokenization. C is involved in training. D is involved in generation but not tokenization."
  },
  {
    "question": "A company wants to reduce the memory footprint of a large language model while preserving most of its performance. Which technique is most appropriate?",
    "question_type": "single_answer",
    "options": {
      "A": "Quantization",
      "B": "Increasing context length",
      "C": "Adding more transformer layers",
      "D": "Using larger embeddings"
    },
    "correct_answer": "A. Quantization",
    "correct_letter": "A",
    "difficulty": "Intermediate",
    "bloom_level": "Apply",
    "tested_concepts": "Quantization; model optimization",
    "evidence_slide_ids": "llm_lecture_2_slide_explanations-S7",
    "why_correct": "Quantization reduces numerical precision and memory requirements while maintaining most model capabilities.",
    "why_each_wrong_option_is_plausible": "B improves context handling, not memory efficiency. C increases memory requirements. D typically increases memory usage."
  },
  {
    "question": "An engineer must choose between two retrieval approaches. One retrieves exact keyword matches while the other retrieves semantically similar content. Which statement best explains the difference?",
    "question_type": "single_answer",
    "options": {
      "A": "Both approaches always return identical results.",
      "B": "Keyword retrieval focuses on lexical matching, while semantic retrieval focuses on meaning.",
      "C": "Semantic retrieval ignores document meaning.",
      "D": "Keyword retrieval requires vector embeddings."
    },
    "correct_answer": "B. Keyword retrieval focuses on lexical matching, while semantic retrieval focuses on meaning.",
    "correct_letter": "B",
    "difficulty": "Difficult",
    "bloom_level": "Analyze",
    "tested_concepts": "Keyword retrieval; semantic retrieval; information retrieval",
    "evidence_slide_ids": "llm_lecture_3_slide_explanations-S12",
    "why_correct": "Keyword retrieval depends on matching terms, whereas semantic retrieval uses representations of meaning.",
    "why_each_wrong_option_is_plausible": "A is incorrect because the methods often return different results. C contradicts the purpose of semantic retrieval. D confuses keyword retrieval with vector search."
  }
]
"""

print("Few-shot examples loaded.")

Few-shot examples loaded.


In [6]:
# Cell 6 — Build Prompt (REWRITTEN — instruction first, lecture second)

def build_prompt(lecture_text, few_shot_examples, num_questions,
                 difficulty_dist, bloom_dist):

    difficulty_str = "\n".join(f"- {k}: {v}" for k, v in difficulty_dist.items())
    bloom_str      = "\n".join(f"- {k}: {v}" for k, v in bloom_dist.items())

    system_msg = "You are an MCQ exam generator. You output only valid JSON arrays. You never output prose or explanations outside the JSON."

    user_msg = f"""TASK: Generate {num_questions} MCQs as a JSON array.

OUTPUT FORMAT — each object must have exactly these fields:
{{
  "question": "...",
  "question_type": "single_answer",
  "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
  "correct_answer": "A. ...",
  "correct_letter": "A",
  "difficulty": "Beginner",
  "bloom_level": "Remember",
  "tested_concepts": "concept1; concept2",
  "evidence_slide_ids": "slide_id_from_notes",
  "why_correct": "explanation",
  "why_each_wrong_option_is_plausible": "A is wrong because... B is wrong because..."
}}

DISTRIBUTION:
Difficulty: {difficulty_str}
Bloom: {bloom_str}

RULES:
- Use ONLY content from the lecture notes below
- Reference real slide IDs in evidence_slide_ids
- Wrong options must reflect real misconceptions
- Output ONLY the JSON array, nothing else

FORMAT EXAMPLE:
{few_shot_examples}

LECTURE NOTES:
{lecture_text}

OUTPUT (JSON array only, start with [ end with ]):"""

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]

    # prompt = tokenizer.apply_chat_template(
    #     messages,
    #     tokenize              = False,
    #     add_generation_prompt = True,
    # )

    # prompt += "\n["
    prompt = system_msg + "\n\n" + user_msg

    return prompt


PROMPT = build_prompt(
    lecture_text      = LECTURE_TEXT,
    few_shot_examples = FEW_SHOT_EXAMPLES,
    num_questions     = NUM_QUESTIONS,
    difficulty_dist   = DIFFICULTY_DIST,
    bloom_dist        = BLOOM_DIST,
)

print(f"Prompt length : {len(PROMPT):,} chars")
print(f"Last 150 chars:")
print(PROMPT[-150:])

Prompt length : 12,164 chars
Last 150 chars:
ction, and 3D shape generation, showing that generative models can improve images, create or predi

OUTPUT (JSON array only, start with [ end with ]):


In [7]:
# Cell 7 — Load Model (FIXED for T4 15GB)
import os, torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except:
    print("No HF token — proceeding without.")

# Use 8-bit instead of 4-bit
# 4-bit saves weights but the attention computation still OOMs
# 8-bit with smaller context is more stable on T4
bnb_config = BitsAndBytesConfig(
    load_in_8bit = True,
)

print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code = True,
    padding_side      = "left",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model in 8-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    low_cpu_mem_usage   = True,
)
model.eval()

torch.cuda.empty_cache()
gc.collect()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
free      = 14.56 - reserved
print(f"Allocated : {allocated:.2f} GB")
print(f"Reserved  : {reserved:.2f} GB")
print(f"Free      : {free:.2f} GB estimated")

No HF token — proceeding without.
Loading tokenizer...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Allocated : 2.00 GB
Reserved  : 2.59 GB
Free      : 11.97 GB estimated


In [8]:
# Cell 8 — Generate (T4 SAFE)
import gc
torch.cuda.empty_cache()
gc.collect()

def generate_mcqs(model, tokenizer, prompt, max_new_tokens,
                  temperature, top_p, repetition_penalty):

    torch.cuda.empty_cache()
    gc.collect()

    inputs = tokenizer(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = 3000,   # hard cap — T4 safe
        padding        = False,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]
    print(f"Input tokens  : {input_len:,}")
    print(f"Output budget : {max_new_tokens:,}")
    print(f"Total budget  : {input_len + max_new_tokens:,}")
    print("Generating...")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_p              = top_p,
            repetition_penalty = repetition_penalty,
            do_sample          = True,
            pad_token_id       = tokenizer.pad_token_id,
            eos_token_id       = tokenizer.eos_token_id,
            use_cache          = True,
        )

    new_tokens = outputs[0][input_len:]
    generated  = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print(f"Generated {len(new_tokens):,} tokens | {len(generated):,} chars")
    return generated


RAW_OUTPUT = generate_mcqs(
    model=model, tokenizer=tokenizer, prompt=PROMPT,
    max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
    top_p=TOP_P, repetition_penalty=REPETITION_PENALTY,
)

with open(f"/kaggle/working/raw_run{RUN_NUMBER}.txt", "w") as f:
    f.write(RAW_OUTPUT)
print("Saved raw output.")
print("\nFirst 800 chars:")
print(RAW_OUTPUT[:800])

Input tokens  : 2,514
Output budget : 4,000
Total budget  : 6,514
Generating...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Generated 288 tokens | 1,370 chars
Saved raw output.

First 800 chars:



```json
[
  {
    "question": "What term describes a branch of science dedicated to creating intelligent machines capable of performing functions normally associated with humans?",
    "question_type": "single_answer",
    "options": {
      "A": "Artificial Intelligence",
      "B": "Machine Learning",
      "C": "Deep Learning",
      "D": "Generative AI"
    },
    "correct_answer": "A. Artificial Intelligence",
    "correct_letter": "A",
    "difficulty": "Beginner",
    "bloom_level": "Remember",
    "tested_concepts": "AI definitions",
    "evidence_slide_ids": "llm_lecture_1_slide_explanations-S1",
    "why_correct": "Artificial Intelligence encompasses technologies designed to mimic human intellectual skills like perception, decision-making, and knowledge representation.", 
    


In [9]:
import json, re

REQUIRED_FIELDS = [
    "question",
    "question_type",
    "options",
    "correct_answer",
    "correct_letter",
    "difficulty",
    "bloom_level",
    "tested_concepts",
    "evidence_slide_ids",
    "why_correct",
    "why_each_wrong_option_is_plausible",
]

VALID_DIFFICULTIES    = {"Beginner", "Intermediate", "Difficult"}
VALID_BLOOM_LEVELS    = {"Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"}
VALID_CORRECT_LETTERS = {"A", "B", "C", "D"}


def extract_json_array(raw_text: str) -> str:
    """
    Robustly extracts a JSON array from raw model output.
    Handles:
      - Output starting with [ ... ]  (perfect case)
      - Output starting with { ... }  (missing opening bracket)
      - Unclosed array (model ran out of tokens)
      - Leading prose before JSON starts
    """
    text = raw_text.strip()

    # ── Find the start of JSON ──
    bracket_pos = text.find("[")
    brace_pos   = text.find("{")

    if brace_pos == -1 and bracket_pos == -1:
        raise ValueError("No JSON found in output. No '[' or '{' detected.")

    # Decide where JSON starts
    if brace_pos != -1 and (bracket_pos == -1 or brace_pos < bracket_pos):
        # Starts with { — wrap in array
        json_str = "[" + text[brace_pos:]
        print("Note: output started with '{' — wrapped in array.")
    else:
        # Starts with [
        json_str = text[bracket_pos:]

    # ── Find the matching closing bracket ──
    depth   = 0
    in_str  = False
    escape  = False
    end_idx = -1

    for i, ch in enumerate(json_str):
        if escape:
            escape = False
            continue
        if ch == "\\" and in_str:
            escape = True
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch in "{[":
            depth += 1
        elif ch in "}]":
            depth -= 1
            if depth == 0:
                end_idx = i
                break

    if end_idx == -1:
        # Array not properly closed — repair by closing after last complete object
        print("WARNING: JSON array not closed. Repairing...")
        last_brace = json_str.rfind("}")
        if last_brace != -1:
            json_str = json_str[:last_brace + 1] + "]"
            print(f"Repaired: closed array after position {last_brace}.")
        else:
            raise ValueError("Cannot repair — no complete JSON object found.")
    else:
        json_str = json_str[:end_idx + 1]
        # Ensure it is wrapped as array
        if not json_str.strip().startswith("["):
            json_str = "[" + json_str
        if not json_str.strip().endswith("]"):
            json_str = json_str + "]"

    # ── Fix common JSON issues ──
    # Remove trailing commas before } or ]
    json_str = re.sub(r",\s*([}\]])", r"\1", json_str)
    # Remove leading commas after [ or {
    json_str = re.sub(r"([{\[])\s*,", r"\1", json_str)

    return json_str


def validate_mcq(mcq: dict, index: int) -> tuple:
    """Validates a single MCQ object. Returns (is_valid, list_of_errors)."""
    errors = []

    # Check all required fields exist and are non-empty
    for field in REQUIRED_FIELDS:
        val = mcq.get(field)
        if val is None or str(val).strip() == "":
            errors.append(f"Missing or empty: '{field}'")

    # Check options A B C D all exist
    if "options" in mcq:
        if not isinstance(mcq["options"], dict):
            errors.append("'options' must be a dict")
        else:
            for letter in ["A", "B", "C", "D"]:
                if letter not in mcq["options"] or not str(mcq["options"][letter]).strip():
                    errors.append(f"Missing option '{letter}'")

    # Check difficulty value
    if mcq.get("difficulty") not in VALID_DIFFICULTIES:
        errors.append(f"Invalid difficulty: '{mcq.get('difficulty')}'")

    # Check bloom level value
    if mcq.get("bloom_level") not in VALID_BLOOM_LEVELS:
        errors.append(f"Invalid bloom_level: '{mcq.get('bloom_level')}'")

    # Check correct letter value
    if mcq.get("correct_letter") not in VALID_CORRECT_LETTERS:
        errors.append(f"Invalid correct_letter: '{mcq.get('correct_letter')}'")

    # Check question type
    if mcq.get("question_type") != "single_answer":
        errors.append(f"question_type must be 'single_answer', got: '{mcq.get('question_type')}'")

    return len(errors) == 0, errors


def extract_complete_questions(raw_text: str) -> list:
    """
    Fallback parser — extracts individual complete JSON objects
    one by one in case the array-level parse fails.
    Useful when the model outputs objects without proper array wrapping.
    """
    text       = "[" + raw_text.strip()
    complete   = []
    depth      = 0
    start      = -1
    in_str     = False
    escape     = False

    for i, ch in enumerate(text):
        if escape:
            escape = False
            continue
        if ch == "\\" and in_str:
            escape = True
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start != -1:
                obj_str = text[start:i + 1]
                # Fix trailing commas
                obj_str = re.sub(r",\s*([}\]])", r"\1", obj_str)
                try:
                    obj = json.loads(obj_str)
                    complete.append(obj)
                except json.JSONDecodeError:
                    pass
                start = -1

    print(f"Object-level parser recovered {len(complete)} objects.")
    return complete


def parse_and_validate(raw_text: str) -> tuple:
    """
    Main parser — tries array-level parse first,
    falls back to object-level extraction if that fails.
    Returns (valid_mcqs, invalid_mcqs).
    """
    mcqs = []

    # ── Attempt 1: Array-level parse ──
    print("Attempt 1: Array-level JSON parse...")
    try:
        json_str = extract_json_array(raw_text)
        mcqs     = json.loads(json_str)
        print(f"Success — parsed {len(mcqs)} objects.")
    except Exception as e:
        print(f"Array-level parse failed: {e}")

        # ── Attempt 2: Fix trailing commas and retry ──
        print("Attempt 2: Fixing trailing commas and retrying...")
        try:
            fixed = re.sub(r",\s*([}\]])", r"\1", "[" + raw_text.strip())
            if not fixed.endswith("]"):
                fixed = fixed[:fixed.rfind("}") + 1] + "]"
            mcqs = json.loads(fixed)
            print(f"Success after fix — parsed {len(mcqs)} objects.")
        except Exception as e2:
            print(f"Fixed parse failed: {e2}")

            # ── Attempt 3: Object-level fallback ──
            print("Attempt 3: Object-level fallback parser...")
            mcqs = extract_complete_questions(raw_text)

    if not mcqs:
        print("ERROR: No MCQ objects could be parsed from output.")
        print("Raw output saved to /kaggle/working/raw_output.txt")
        return [], []

    print(f"\nTotal objects parsed : {len(mcqs)}")

    # ── Validate each MCQ ──
    valid_mcqs   = []
    invalid_mcqs = []

    for i, mcq in enumerate(mcqs):
        if not isinstance(mcq, dict):
            invalid_mcqs.append({"index": i, "errors": ["Not a dict"], "mcq": mcq})
            continue
        is_valid, errors = validate_mcq(mcq, i)
        if is_valid:
            valid_mcqs.append(mcq)
        else:
            invalid_mcqs.append({"index": i, "errors": errors, "mcq": mcq})
            print(f"  Q{i+1} INVALID: {errors}")

    return valid_mcqs, invalid_mcqs


# ── Run parsing ──
VALID_MCQS, INVALID_MCQS = parse_and_validate(RAW_OUTPUT)

print(f"\n{'='*50}")
print(f"Valid MCQs   : {len(VALID_MCQS)}")
print(f"Invalid MCQs : {len(INVALID_MCQS)}")
print(f"Target       : {NUM_QUESTIONS}")
print(f"{'='*50}")

# ── Show first valid question as sanity check ──
if VALID_MCQS:
    q = VALID_MCQS[0]
    print(f"\nSample — Question 1:")
    print(f"  Q          : {q['question']}")
    for letter in ["A", "B", "C", "D"]:
        marker = "✓" if letter == q.get("correct_letter") else " "
        print(f"  {marker} {letter}        : {q['options'].get(letter, '')}")
    print(f"  Difficulty : {q['difficulty']}")
    print(f"  Bloom      : {q['bloom_level']}")
    print(f"  Slide ref  : {q['evidence_slide_ids']}")
    print(f"  Why correct: {q['why_correct'][:100]}")
else:
    print("\nNo valid MCQs found.")
    print("Check /kaggle/working/raw_output.txt for raw model output.")

Attempt 1: Array-level JSON parse...
Repaired: closed array after position 1327.
Success — parsed 1 objects.

Total objects parsed : 1
  Q1 INVALID: ["Missing or empty: 'why_each_wrong_option_is_plausible'"]

Valid MCQs   : 0
Invalid MCQs : 1
Target       : 10

No valid MCQs found.
Check /kaggle/working/raw_output.txt for raw model output.


In [10]:
# ── Repair invalid MCQs — fill missing fields ──
import torch, gc

def repair_invalid_mcqs(invalid_mcqs: list, model, tokenizer) -> list:
    repaired = []

    for item in invalid_mcqs:
        mcq    = item["mcq"]
        errors = item["errors"]
        print(f"\nRepairing Q{item['index']+1}: {errors}")

        # Build a short repair prompt
        missing_fields = [e.replace("Missing or empty: '", "").replace("'", "")
                          for e in errors if "Missing" in e]

        repair_prompt = f"""Complete these missing fields for this MCQ and return ONLY valid JSON for the complete object.

INCOMPLETE MCQ:
{json.dumps(mcq, indent=2)}

MISSING FIELDS TO ADD: {missing_fields}

Rules:
- why_correct: explain why the correct answer ({mcq.get('correct_answer','')}) is right
- why_each_wrong_option_is_plausible: explain why each wrong option seems plausible
- tested_concepts: list 2-3 key concepts being tested, separated by semicolons
- Return the COMPLETE JSON object with ALL fields filled
- Output ONLY the JSON object, nothing else"""

        messages = [
            {"role": "system", "content": "You complete incomplete JSON objects. Output only valid JSON."},
            {"role": "user",   "content": repair_prompt},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ) + "\n{"

        torch.cuda.empty_cache()
        gc.collect()

        inputs = tokenizer(prompt, return_tensors="pt",
                          truncation=True, max_length=1500).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens     = 500,
                temperature        = 0.3,
                do_sample          = True,
                pad_token_id       = tokenizer.pad_token_id,
                eos_token_id       = tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        result     = "{" + tokenizer.decode(new_tokens, skip_special_tokens=True)

        # Extract JSON object
        try:
            last_brace = result.rfind("}")
            obj_str    = result[:last_brace + 1]
            obj_str    = re.sub(r",\s*([}\]])", r"\1", obj_str)
            repaired_mcq = json.loads(obj_str)
            is_valid, errors_after = validate_mcq(repaired_mcq, 0)
            if is_valid:
                print(f"  ✅ Repaired successfully.")
                repaired.append(repaired_mcq)
            else:
                print(f"  ⚠️  Still invalid after repair: {errors_after}")
                # Merge missing fields manually
                for field in missing_fields:
                    if field in repaired_mcq and repaired_mcq[field]:
                        mcq[field] = repaired_mcq[field]
                is_valid2, _ = validate_mcq(mcq, 0)
                if is_valid2:
                    print(f"  ✅ Merged missing fields successfully.")
                    repaired.append(mcq)
        except Exception as e:
            print(f"  ❌ Could not repair: {e}")

    return repaired


REPAIRED = repair_invalid_mcqs(INVALID_MCQS, model, tokenizer)
VALID_MCQS.extend(REPAIRED)

print(f"\nAfter repair:")
print(f"  Valid MCQs : {len(VALID_MCQS)}")
print(f"  Target     : {NUM_QUESTIONS}")


Repairing Q1: ["Missing or empty: 'why_each_wrong_option_is_plausible'"]
  ✅ Repaired successfully.

After repair:
  Valid MCQs : 1
  Target     : 10


In [11]:
# Quick check — does the output contain any JSON?
import re

print("=== RAW OUTPUT ANALYSIS ===")
print(f"Total chars : {len(RAW_OUTPUT):,}")
print(f"Contains '[' : {'[' in RAW_OUTPUT}")
print(f"Contains 'question' : {'question' in RAW_OUTPUT}")
print(f"Contains 'options' : {'options' in RAW_OUTPUT}")
print(f"\nFirst 300 chars:")
print(RAW_OUTPUT[:300])
print(f"\nLast 300 chars:")
print(RAW_OUTPUT[-300:])

# Find first '['
bracket_pos = RAW_OUTPUT.find('[')
brace_pos   = RAW_OUTPUT.find('{')
print(f"\nFirst '[' at position : {bracket_pos}")
print(f"First '{{' at position : {brace_pos}")
if bracket_pos != -1:
    print(f"\nContent from first '[' (300 chars):")
    print(RAW_OUTPUT[bracket_pos:bracket_pos+300])

=== RAW OUTPUT ANALYSIS ===
Total chars : 1,370
Contains '[' : True
Contains 'question' : True
Contains 'options' : True

First 300 chars:



```json
[
  {
    "question": "What term describes a branch of science dedicated to creating intelligent machines capable of performing functions normally associated with humans?",
    "question_type": "single_answer",
    "options": {
      "A": "Artificial Intelligence",
      "B": "Machine Lea

Last 300 chars:
ction \nand pattern recognition across vast amounts of structured data sets\n\n While D concentrates primarily upon developing techniques aimed towards producing novel creations e.g., images ,music etc . It doesn't cover everything classified broadly under A."
    
  },{
     ...
   ]
```

Note: Due

First '[' at position : 11
First '{' at position : 15

Content from first '[' (300 chars):
[
  {
    "question": "What term describes a branch of science dedicated to creating intelligent machines capable of performing functions normally ass

In [12]:
# ── Diagnostic — see what the model actually output ──
print(f"Total chars : {len(RAW_OUTPUT):,}")
print(f"Contains 'question' : {'question' in RAW_OUTPUT}")
print(f"Contains 'options'  : {'options' in RAW_OUTPUT}")
print(f"Contains 'correct'  : {'correct' in RAW_OUTPUT}")

# Count how many questions were started
q_count = RAW_OUTPUT.count('"question"')
print(f"\nQuestions started : {q_count}")

# Count how many were completed
complete = RAW_OUTPUT.count('"why_each_wrong_option_is_plausible"')
print(f"Questions completed : {complete}")

print(f"\nFirst 500 chars:")
print("-" * 60)
print(RAW_OUTPUT[:500])
print("\nLast 500 chars:")
print("-" * 60)
print(RAW_OUTPUT[-500:])

Total chars : 1,370
Contains 'question' : True
Contains 'options'  : True
Contains 'correct'  : True

Questions started : 1
Questions completed : 0

First 500 chars:
------------------------------------------------------------



```json
[
  {
    "question": "What term describes a branch of science dedicated to creating intelligent machines capable of performing functions normally associated with humans?",
    "question_type": "single_answer",
    "options": {
      "A": "Artificial Intelligence",
      "B": "Machine Learning",
      "C": "Deep Learning",
      "D": "Generative AI"
    },
    "correct_answer": "A. Artificial Intelligence",
    "correct_letter": "A",
    "difficulty": "Beginner",
    "bloom_level": "R

Last 500 chars:
------------------------------------------------------------
ch focusing solely on algorithms that adapt based on experience without explicit programming.\n\n C specifies a particular method under ML where multiple layers of neurons enable complex feature

In [13]:
# ─────────────────────────────────────────────────────────────────
# Cell 10 — Analyze Distribution
# ─────────────────────────────────────────────────────────────────

def analyze_distribution(mcqs: list) -> None:
    if not mcqs:
        print("No valid MCQs to analyze.")
        return

    diff_counts  = Counter(q.get("difficulty",  "Unknown") for q in mcqs)
    bloom_counts = Counter(q.get("bloom_level", "Unknown") for q in mcqs)

    print(f"\n{'='*55}")
    print(f"DISTRIBUTION ANALYSIS — {len(mcqs)} questions")
    print(f"{'='*55}")

    print("\nDifficulty:")
    for level in ["Beginner", "Intermediate", "Difficult"]:
        count  = diff_counts.get(level, 0)
        target = DIFFICULTY_DIST.get(level, 0)
        bar    = "█" * min(count, 40)
        flag   = "✅" if count == target else f"⚠️  target={target}"
        print(f"  {level:15s} {count:3d}  {bar:40s}  {flag}")

    print("\nBloom's Taxonomy:")
    for level in ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]:
        count  = bloom_counts.get(level, 0)
        target = BLOOM_DIST.get(level, 0)
        bar    = "█" * min(count, 40)
        flag   = "✅" if count == target else f"⚠️  target={target}"
        print(f"  {level:15s} {count:3d}  {bar:40s}  {flag}")

    print("\nSample — Question 1:")
    q = mcqs[0]
    print(f"  Q   : {q['question'][:120]}")
    for letter in ["A", "B", "C", "D"]:
        marker = "✓" if letter == q.get("correct_letter") else " "
        print(f"  {marker} {letter} : {q['options'].get(letter, '')}")
    print(f"  Difficulty  : {q['difficulty']}")
    print(f"  Bloom       : {q['bloom_level']}")
    print(f"  Slide ref   : {q['evidence_slide_ids']}")
    print(f"  Why correct : {q['why_correct'][:100]}")


analyze_distribution(VALID_MCQS)


DISTRIBUTION ANALYSIS — 1 questions

Difficulty:
  Beginner          1  █                                         ⚠️  target=3
  Intermediate      0                                            ⚠️  target=4
  Difficult         0                                            ⚠️  target=3

Bloom's Taxonomy:
  Remember          1  █                                         ⚠️  target=2
  Understand        0                                            ⚠️  target=2
  Apply             0                                            ⚠️  target=2
  Analyze           0                                            ⚠️  target=2
  Evaluate          0                                            ⚠️  target=1
  Create            0                                            ⚠️  target=1

Sample — Question 1:
  Q   : What term describes a branch of science dedicated to creating intelligent machines capable of performing functions norma
  ✓ A : Artificial Intelligence
    B : Machine Learning
    C : Deep Learning

In [14]:
# ─────────────────────────────────────────────────────────────────
# Cell 11 — Save Output
# ─────────────────────────────────────────────────────────────────

def save_output(valid_mcqs: list, invalid_mcqs: list, output_path: str) -> None:

    output = {
        "metadata": {
            "model"             : MODEL_NAME,
            "num_questions"     : len(valid_mcqs),
            "target_questions"  : NUM_QUESTIONS,
            "difficulty_dist"   : DIFFICULTY_DIST,
            "bloom_dist"        : BLOOM_DIST,
            "lecture_source"    : LECTURE_INPUT_DIR,
            "slides_loaded"     : len(SLIDE_IDS),
            "lecture_chars_used": len(LECTURE_TEXT),
        },
        "mcqs": valid_mcqs,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved {len(valid_mcqs)} valid MCQs → {output_path}")
    print(f"   File size: {Path(output_path).stat().st_size / 1024:.1f} KB")

    if invalid_mcqs:
        inv_path = output_path.replace(".json", "_invalid.json")
        with open(inv_path, "w", encoding="utf-8") as f:
            json.dump(invalid_mcqs, f, indent=2, ensure_ascii=False)
        print(f"⚠️  Saved {len(invalid_mcqs)} invalid MCQs → {inv_path}")


save_output(VALID_MCQS, INVALID_MCQS, OUTPUT_PATH)

print(f"\n{'='*55}")
print(f"MCQ GENERATION COMPLETE")
print(f"{'='*55}")
print(f"Model          : {MODEL_NAME}")
print(f"Valid MCQs     : {len(VALID_MCQS)} / {NUM_QUESTIONS}")
print(f"Success rate   : {len(VALID_MCQS)/NUM_QUESTIONS*100:.1f}%")
print(f"Output file    : {OUTPUT_PATH}")

✅ Saved 1 valid MCQs → /kaggle/working/mcqs_run1.json
   File size: 1.8 KB
⚠️  Saved 1 invalid MCQs → /kaggle/working/mcqs_run1_invalid.json

MCQ GENERATION COMPLETE
Model          : Qwen/Qwen3-4B-Base
Valid MCQs     : 1 / 10
Success rate   : 10.0%
Output file    : /kaggle/working/mcqs_run1.json


In [15]:
# ── AUTO LOOP — Runs 2 to 15 automatically ──
# Make sure Cell 11 already saved Run 1 before running this

import json, os, gc, torch

TOTAL_RUNS = 15

for run in range(2, TOTAL_RUNS + 1):

    print(f"\n{'='*60}")
    print(f"RUN {run}/{TOTAL_RUNS}")
    print(f"{'='*60}")

    # ── Update run settings ──
    RUN_NUMBER  = run
    SLICE_START = (run - 1) * 8_000
    SLICE_END   =  run      * 8_000
    OUTPUT_PATH = f"/kaggle/working/mcqs_run{run}.json"

    print(f"Chars: {SLICE_START:,} → {SLICE_END:,}")

    # ── Reload lecture slice ──
    full_lines = []
    import glob
    from pathlib import Path

    json_files = sorted(glob.glob(
        "/kaggle/input/datasets/mayarboghdady/lec-nots/**/*.json",
        recursive=True
    ))

    all_parts = []
    all_ids   = []
    for fpath in json_files:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        fname = Path(fpath).stem
        if isinstance(data, list):
            all_parts.append(f"\n=== {fname} ===\n")
            for i, item in enumerate(data):
                if isinstance(item, dict):
                    sid     = item.get("slide_id") or f"{fname}-S{i+1}"
                    title   = item.get("slide_title") or item.get("title") or ""
                    content = item.get("slide_paragraph") or item.get("content") or str(item)
                    all_ids.append(str(sid))
                    part = f"[Slide {sid}]"
                    if title and title != "None":
                        part += f" {title}\n"
                    part += f"\n{content}\n"
                    all_parts.append(part)

    full_text    = "\n".join(all_parts)
    LECTURE_TEXT = full_text[SLICE_START:SLICE_END]

    if len(LECTURE_TEXT) == 0:
        print(f"Run {run}: empty slice — skipping.")
        continue

    print(f"Lecture slice: {len(LECTURE_TEXT):,} chars")

    # ── Rebuild prompt ──
    PROMPT = build_prompt(
        lecture_text      = LECTURE_TEXT,
        few_shot_examples = FEW_SHOT_EXAMPLES,
        num_questions     = NUM_QUESTIONS,
        difficulty_dist   = DIFFICULTY_DIST,
        bloom_dist        = BLOOM_DIST,
    )

    # ── Generate ──
    torch.cuda.empty_cache()
    gc.collect()

    RAW_OUTPUT = generate_mcqs(
        model              = model,
        tokenizer          = tokenizer,
        prompt             = PROMPT,
        max_new_tokens     = MAX_NEW_TOKENS,
        temperature        = TEMPERATURE,
        top_p              = TOP_P,
        repetition_penalty = REPETITION_PENALTY,
    )

    # Save raw output
    with open(f"/kaggle/working/raw_run{run}.txt", "w") as f:
        f.write(RAW_OUTPUT)

    # ── Parse and validate ──
    valid, invalid = parse_and_validate(RAW_OUTPUT)
    print(f"Valid: {len(valid)} | Invalid: {len(invalid)}")

    # ── Save run output ──
    output = {
        "metadata": {
            "run"           : run,
            "model"         : MODEL_NAME,
            "num_questions" : len(valid),
            "slice_start"   : SLICE_START,
            "slice_end"     : SLICE_END,
        },
        "mcqs": valid
    }
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(valid)} questions → {OUTPUT_PATH}")

    # Clear memory between runs
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*60}")
print("ALL RUNS COMPLETE")
print(f"{'='*60}")


RUN 2/15
Chars: 8,000 → 16,000
Lecture slice: 8,000 chars
Input tokens  : 2,489
Output budget : 4,000
Total budget  : 6,489
Generating...
Generated 203 tokens | 1,012 chars
Attempt 1: Array-level JSON parse...
Repaired: closed array after position 951.
Array-level parse failed: Expecting property name enclosed in double quotes: line 2 column 82 (char 83)
Attempt 2: Fixing trailing commas and retrying...
Fixed parse failed: Expecting property name enclosed in double quotes: line 2 column 82 (char 84)
Attempt 3: Object-level fallback parser...
Object-level parser recovered 0 objects.
ERROR: No MCQ objects could be parsed from output.
Raw output saved to /kaggle/working/raw_output.txt
Valid: 0 | Invalid: 0
Saved 0 questions → /kaggle/working/mcqs_run2.json

RUN 3/15
Chars: 16,000 → 24,000
Lecture slice: 8,000 chars
Input tokens  : 2,579
Output budget : 4,000
Total budget  : 6,579
Generating...
Generated 574 tokens | 3,416 chars
Attempt 1: Array-level JSON parse...
Repaired: closed array 

In [16]:
# ── AUTO LOOP — Runs 2 to 15 automatically ──
# Make sure Cell 11 already saved Run 1 before running this

import json, os, gc, torch

TOTAL_RUNS = 15

for run in range(2, TOTAL_RUNS + 1):

    print(f"\n{'='*60}")
    print(f"RUN {run}/{TOTAL_RUNS}")
    print(f"{'='*60}")

    # ── Update run settings ──
    RUN_NUMBER  = run
    SLICE_START = (run - 1) * 8_000
    SLICE_END   =  run      * 8_000
    OUTPUT_PATH = f"/kaggle/working/mcqs_run{run}.json"

    print(f"Chars: {SLICE_START:,} → {SLICE_END:,}")

    # ── Reload lecture slice ──
    full_lines = []
    import glob
    from pathlib import Path

    json_files = sorted(glob.glob(
        "/kaggle/input/datasets/mayarboghdady/lec-nots/**/*.json",
        recursive=True
    ))

    all_parts = []
    all_ids   = []
    for fpath in json_files:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        fname = Path(fpath).stem
        if isinstance(data, list):
            all_parts.append(f"\n=== {fname} ===\n")
            for i, item in enumerate(data):
                if isinstance(item, dict):
                    sid     = item.get("slide_id") or f"{fname}-S{i+1}"
                    title   = item.get("slide_title") or item.get("title") or ""
                    content = item.get("slide_paragraph") or item.get("content") or str(item)
                    all_ids.append(str(sid))
                    part = f"[Slide {sid}]"
                    if title and title != "None":
                        part += f" {title}\n"
                    part += f"\n{content}\n"
                    all_parts.append(part)

    full_text    = "\n".join(all_parts)
    LECTURE_TEXT = full_text[SLICE_START:SLICE_END]

    if len(LECTURE_TEXT) == 0:
        print(f"Run {run}: empty slice — skipping.")
        continue

    print(f"Lecture slice: {len(LECTURE_TEXT):,} chars")

    # ── Rebuild prompt ──
    PROMPT = build_prompt(
        lecture_text      = LECTURE_TEXT,
        few_shot_examples = FEW_SHOT_EXAMPLES,
        num_questions     = NUM_QUESTIONS,
        difficulty_dist   = DIFFICULTY_DIST,
        bloom_dist        = BLOOM_DIST,
    )

    # ── Generate ──
    torch.cuda.empty_cache()
    gc.collect()

    RAW_OUTPUT = generate_mcqs(
        model              = model,
        tokenizer          = tokenizer,
        prompt             = PROMPT,
        max_new_tokens     = MAX_NEW_TOKENS,
        temperature        = TEMPERATURE,
        top_p              = TOP_P,
        repetition_penalty = REPETITION_PENALTY,
    )

    # Save raw output
    with open(f"/kaggle/working/raw_run{run}.txt", "w") as f:
        f.write(RAW_OUTPUT)

    # ── Parse and validate ──
    valid, invalid = parse_and_validate(RAW_OUTPUT)
    print(f"Valid: {len(valid)} | Invalid: {len(invalid)}")

    # ── Save run output ──
    output = {
        "metadata": {
            "run"           : run,
            "model"         : MODEL_NAME,
            "num_questions" : len(valid),
            "slice_start"   : SLICE_START,
            "slice_end"     : SLICE_END,
        },
        "mcqs": valid
    }
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(valid)} questions → {OUTPUT_PATH}")

    # Clear memory between runs
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*60}")
print("ALL RUNS COMPLETE")
print(f"{'='*60}")


RUN 2/15
Chars: 8,000 → 16,000
Lecture slice: 8,000 chars
Input tokens  : 2,489
Output budget : 4,000
Total budget  : 6,489
Generating...
Generated 479 tokens | 2,258 chars
Attempt 1: Array-level JSON parse...
Repaired: closed array after position 1559.
Array-level parse failed: Expecting ',' delimiter: line 28 column 6 (char 1560)
Attempt 2: Fixing trailing commas and retrying...
Fixed parse failed: Expecting value: line 1 column 2 (char 1)
Attempt 3: Object-level fallback parser...
Object-level parser recovered 1 objects.

Total objects parsed : 1
Valid: 1 | Invalid: 0
Saved 1 questions → /kaggle/working/mcqs_run2.json

RUN 3/15
Chars: 16,000 → 24,000
Lecture slice: 8,000 chars
Input tokens  : 2,579
Output budget : 4,000
Total budget  : 6,579
Generating...
Generated 909 tokens | 2,378 chars
Attempt 1: Array-level JSON parse...
Repaired: closed array after position 2372.
Array-level parse failed: Expecting property name enclosed in double quotes: line 2 column 88 (char 89)
Attempt 2: